In [195]:
# --- INSTALLATION ---
# python -m venv .venv
# .venv\Scripts\activate
# pip install ipykernel pandas openpyxl

<h1>Setting Up</h1>

In [196]:
# --- Import Libraries ---
import pandas as pd
import datetime

In [197]:
# --- Import Data ---
data = pd.read_csv(r'Airbnb_Open_Data.csv')

print(data.columns)
print(data.info())
display(data.head())

C:\Users\THU THAO\AppData\Local\Temp\ipykernel_152272\2292410139.py:2: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(r'Airbnb_Open_Data.csv')


Index(['id', 'NAME', 'host id', 'host_identity_verified', 'host name',
       'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country',
       'country code', 'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'last review', 'reviews per month',
       'review rate number', 'calculated host listings count',
       'availability 365', 'house_rules', 'license'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  str    
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  str    
 4   host name                     

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


<h1> Data Cleaning </h1>

In [198]:
# --- Dropping Rebundance Columns ---
cols_to_keep = [ 'NAME', 'host id', 'host_identity_verified', 'host name',
                'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country',
                'country code', 'instant_bookable', 'cancellation_policy', 'room type',
                'Construction year', 'price', 'service fee', 'minimum nights',
                'number of reviews', 'last review']
cols_to_drop = [ 'reviews per month', 'review rate number', 'calculated host listings count',
                'availability 365', 'house_rules', 'license', 'id' ]

# 1. Using Filtering
df = data[cols_to_keep]

# 2. Using Columns Dropping
data = data.drop(columns=cols_to_drop)

In [199]:
# --- Renaming Columns ---
for col in data.columns:
    data = data.rename(columns={col: col.strip().replace(' ', '_').lower()})

In [200]:
# --- Dropping Duplicates ---
print(data.duplicated().value_counts())

data = data.drop_duplicates()

False    102058
True        541
Name: count, dtype: int64


In [201]:
# --- Remove NaN Values ---
print(data.isna().sum())

# lat, long, price, neighbourhood, neighbourhood_group: drop rows
data = data.dropna(subset=['lat', 'long', 'price', 'neighbourhood', 'neighbourhood_group'])

# number_of_reviews: convert to 0
data['number_of_reviews'] = data['number_of_reviews'].fillna(0)

# last_review: convert column to days_since_last_review, NaN to 9999 days
data['last_review'] = pd.to_datetime(data['last_review'], errors='coerce')
today = pd.to_datetime(datetime.date.today())
data['days_since_last_review'] = (today - data['last_review']).dt.days
data['days_since_last_review'] = data['days_since_last_review'].fillna(9999)
# drop last_review column
data = data.drop(columns=['last_review'])

# name, host_name: convert to 'Unknown'
data['name'] = data['name'].fillna('Unknown')
data['host_name'] = data['host_name'].fillna('Unknown')

# host_identity_verified: fill with verified value of same host if host has been verified before
data['host_identity_verified'] = data['host_identity_verified'].str.lower()
verified = (data['host_identity_verified'] == 'verified').groupby(data['host_id']).transform('any').map({True: 'verified', False: 'unconfirmed'})
data['host_identity_verified'] = data['host_identity_verified'].fillna(verified)

# instant_bookable: convert to False
data['instant_bookable'] = data['instant_bookable'].fillna(False)

# cancellation_policy: fill with mode
data['cancellation_policy'] = data['cancellation_policy'].fillna(df['cancellation_policy'].mode()[0])

# service_fee, minimum_nights, construction_year: fill with median value
# create new column for service_fee without symbols
data['service_fee_num'] = pd.to_numeric(data['service_fee'].astype(str)
                                            .str.replace('$', '', regex=False)
                                            .str.replace(',', '', regex=False))
for col in ['service_fee_num', 'minimum_nights', 'construction_year']:
    data[col] = data[col].fillna(data[col].median())
# convert column into service_fee
data = data.drop(columns='service_fee')
data = data.rename(columns={'service_fee_num': 'service_fee'})

# country: fill with non NaN value of same neighbourhood_group
data.groupby('neighbourhood_group')['country'].ffill().bfill()

# country_code: fill with non NaN value of same country
data.groupby('country')['country_code'].ffill().bfill()

# drop all leftover NaN rows
data = data.dropna()
data = data.reset_index(drop=True)

name                        250
host_id                       0
host_identity_verified      289
host_name                   404
neighbourhood_group          29
neighbourhood                16
lat                           8
long                          8
country                     532
country_code                131
instant_bookable            105
cancellation_policy          76
room_type                     0
construction_year           214
price                       247
service_fee                 273
minimum_nights              400
number_of_reviews           183
last_review               15832
dtype: int64


In [202]:
# --- Changing Individual Columns ---
# host_identity_verified: change into uppercase
data['host_identity_verified'] = data['host_identity_verified'].str.upper()

# instant_bookable: change into 1 and 0
data['instant_bookable'] = data['instant_bookable'].apply(lambda x: 1 if x == True else 0)

# price: convert to num
data['price'] = pd.to_numeric(data['price'].astype(str)
                                .str.replace('$', '', regex=False)
                                .str.replace(',', '', regex=False))

# construction_year, minimum_nights, number_of_reviews, days_since_last_review: convert to int
for col in ['construction_year', 'minimum_nights', 'number_of_reviews', 'days_since_last_review']:
    data[col] = data[col].astype(int)

display(data)

,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,country_code,instant_bookable,cancellation_policy,room_type,construction_year,price,minimum_nights,number_of_reviews,days_since_last_review,service_fee
0,Clean & quiet apt home by the park,80014485718,UNCONFIRMED,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,US,0,strict,Private room,2020,966,10,9,1797,193.0
1,Skylit Midtown Castle,52335172823,VERIFIED,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,US,0,moderate,Entire home/apt,2007,142,30,45,1583,28.0
2,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,UNCONFIRMED,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,US,1,flexible,Private room,2005,620,3,0,9999,124.0
3,Unknown,85098326012,UNCONFIRMED,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,US,1,moderate,Entire home/apt,2005,368,30,270,2634,74.0
4,Entire Apt: Spacious Studio/Loft by central park,92037596077,VERIFIED,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,US,0,moderate,Entire home/apt,2009,204,10,9,2862,41.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101187,Cozy bright room near Prospect Park,77326652202,UNCONFIRMED,Mariam,Brooklyn,Flatbush,40.64945,-73.96108,United States,US,1,moderate,Private room,2012,696,7,12,2734,125.0
101188,Private Bedroom with Amazing Rooftop View,45936254757,VERIFIED,Trey,Brooklyn,Bushwick,40.69872,-73.92718,United States,US,0,flexible,Private room,2012,909,1,19,3307,125.0
101189,Pretty Brooklyn One-Bedroom for 2 to 4 people,23801060917,VERIFIED,Michael,Brooklyn,Bedford-Stuyvesant,40.67810,-73.90822,United States,US,1,moderate,Entire home/apt,2012,387,2,50,2643,125.0
101190,Room & private bathroom in historic Harlem,15593031571,UNCONFIRMED,Shireen,Manhattan,Harlem,40.81248,-73.94317,United States,US,1,strict,Private room,2012,848,2,0,9999,125.0


In [203]:
# --- Exporting ---
data.to_csv('Airbnb_Cleaned_Data.csv', index=False)

data.to_excel('Airbnb_Cleaned_Data.xlsx', index=False)